# Web Technology — 20 measured experiments (W01–W20)

Companion to the [Web Technology course](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md). The notebook runs the real teaching app — browser files, FastAPI API and SQLite database — in three ways:

* **in-process** with FastAPI's `TestClient` (fast, no network);
* as a **live Uvicorn server** on 127.0.0.1 inside this runtime (real HTTP over TCP);
* in a **real Chromium browser** driven by Playwright (W05, W06, W19).

Each section: a short concept recap with a link to the lesson, a **See it** cell you can edit, and a **Measure it** cell that produces a chart, a CSV and a JSON file. Run **Runtime → Run all** (≈ 3 minutes including the one-off browser install).

In [ ]:
# Setup: get the repository and the lab's Python dependencies.
import os, subprocess, sys
from pathlib import Path
if Path('web_lab/app.py').exists():
    ROOT = Path('.')
elif Path('app.py').exists() and Path('static').exists():
    ROOT = Path('..')
else:
    if not Path('ATLAS').exists():
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/FUllSCIPTS/ATLAS.git'], check=True)
    ROOT = Path('ATLAS')
os.chdir(ROOT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'web_lab/requirements.txt'], check=True)
sys.path.insert(0, str(Path('.').resolve()))
print('working in', Path('.').resolve())

In [ ]:
# One-off: a headless Chromium for the three browser experiments (W05, W06, W19).
# Set INSTALL_BROWSER = False to skip them.
INSTALL_BROWSER = True
BROWSER_OK = False
if INSTALL_BROWSER:
    try:
        import playwright  # noqa: F401
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'playwright'], check=True)
    r = subprocess.run([sys.executable, '-m', 'playwright', 'install', '--with-deps', 'chromium'],
                       capture_output=True, text=True)
    if r.returncode != 0:            # e.g. no root for system packages: try without --with-deps
        r = subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'], capture_output=True, text=True)
    BROWSER_OK = r.returncode == 0
print('browser experiments enabled:', BROWSER_OK)

In [ ]:
import json
from IPython.display import HTML, Image, display
from web_lab import experiments as wl
OUT = Path('web_results'); OUT.mkdir(exist_ok=True)

def lab(n):
    if n in wl.BROWSER and not BROWSER_OK:
        print(f'W{n:02} skipped: browser not installed'); return None
    try:
        res = wl.run(n, OUT)
    except wl.BrowserUnavailable as e:
        print(f'W{n:02} skipped:', e); return None
    png = wl.draw(res, OUT)
    print(f"W{n:02} {res['title']}")
    print('  ', res['xlabel'], '|', ' | '.join(res['labels']))
    for row in res['rows']:
        print('  ', ' | '.join(f'{v:.4g}' if isinstance(v, float) else str(v) for v in row))
    if res['notes']:
        print('Note:', res['notes'])
    display(Image(filename=str(png)))
    return res

# First, the integration tests: a failure here must be investigated, never ignored.
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', '--color=no', 'web_lab/tests'],
                   capture_output=True, text=True)
print(r.stdout[-600:])

## W01 — HTTP is text over TCP

**Concept.** A request is a request line, headers, a blank line and an optional body; so is a response. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-3--http-requests-responses-methods-status-codes-and-headers).

**See it:**

In [ ]:
from urllib.parse import urlsplit, parse_qs
u = urlsplit('https://rides.example.ie:8443/api/rides?status=requested&limit=20#top')
print(u.scheme, u.hostname, u.port, u.path, parse_qs(u.query), u.fragment)

**Measure it.** Predict first, then run. *What to look for:* After running W01, print the raw exchange: `print(res1['extra']['raw_http'])`.

In [ ]:
res1 = lab(1)

## W02 — Status codes

**Concept.** 2xx success, 3xx not modified/redirect, 4xx the client must change something, 5xx server failure. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-3--http-requests-responses-methods-status-codes-and-headers).

**See it:**

In [ ]:
from fastapi.testclient import TestClient
from web_lab.app import create_app
import tempfile
with TestClient(create_app(Path(tempfile.mkdtemp()) / 'demo.sqlite3')) as c:
    r = c.post('/api/rides', json={'passenger': 'Student', 'pickup': 'UCC', 'destination': 'Kent',
                                   'distance_km': 4.5, 'request_key': 'notebook-key-1'})
    print(r.status_code, json.dumps(r.json(), indent=1)[:300])

**Measure it.** Predict first, then run. *What to look for:* Which codes would a careful client retry automatically, and which never?

In [ ]:
res2 = lab(2)

## W03 — HTML is a tree

**Concept.** The browser parses HTML into the DOM tree; every form control needs a label. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-5--html-documents-elements-attributes-and-the-tree).

**See it:**

In [ ]:
display(HTML('''<main style="font-family:sans-serif"><h3>Cork rides</h3>
<p>Fares: <strong>€2.50</strong> + <em>€1.50/km</em></p>
<label for="p">Pickup</label> <input id="p" name="pickup" value="UCC"></main>'''))
print(Path('web_lab/static/index.html').read_text()[:600])

**Measure it.** Predict first, then run. *What to look for:* Count the elements by hand for the small snippet, then compare with W03's chart for the real page.

In [ ]:
res3 = lab(3)

## W04 — CSS specificity and the box model

**Concept.** Specificity (ids, classes, elements) decides which rule wins; `box-sizing` decides what `width` means. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-7--css-rules-selectors-specificity-and-the-cascade).

**See it:**

In [ ]:
display(HTML('''<div style="font-family:sans-serif">
<div style="width:220px;padding:20px;border:6px solid #135da6;margin:10px;background:#e8f0fe;box-sizing:content-box">
content-box: rendered 272 px wide</div>
<div style="width:220px;padding:20px;border:6px solid #135da6;margin:10px;background:#e8f0fe;box-sizing:border-box">
border-box: rendered 220 px wide</div></div>'''))
for sel in ['#message', '.ride strong', 'input:invalid:not(:placeholder-shown)', 'button:hover']:
    print(f'{sel:<40} specificity {wl.specificity(sel)}')

**Measure it.** Predict first, then run. *What to look for:* Why does `#message` beat every class selector, whatever the order?

In [ ]:
res4 = lab(4)

## W05 — Responsive layout

**Concept.** A media query switches the grid from two columns to one below 801 px. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-9--responsive-and-accessible-design).

**See it:**

In [ ]:
display(HTML('''<div style="display:grid;grid-template-columns:repeat(2,minmax(0,1fr));gap:8px;font-family:sans-serif">
<div style="background:#e8f0fe;padding:12px">panel 1</div><div style="background:#d7f5dd;padding:12px">panel 2</div></div>'''))
print([l for l in Path('web_lab/static/style.css').read_text().splitlines() if '@media' in l or 'grid-template' in l])

**Measure it.** Predict first, then run. *What to look for:* Open the two screenshots after the run: `display(Image(OUT/'W05_viewport_480.png', width=300))`.

In [ ]:
res5 = lab(5)

## W06 — XSS and safe rendering

**Concept.** `textContent` shows text; `innerHTML` parses — and may run — it. The CSP header is a second layer. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-11--the-dom-events-and-safe-rendering).

**See it:**

In [ ]:
print('payload typed into the passenger field:', wl.XSS)
print([l.strip() for l in Path('web_lab/static/app.js').read_text().splitlines() if 'textContent' in l][:3])

**Measure it.** Predict first, then run. *What to look for:* The first bar must be 0. The third shows why the second layer (CSP) matters.

In [ ]:
res6 = lab(6)

## W07 — Server-side validation

**Concept.** Pydantic rejects bad bodies with 422 and names every faulty field. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-14--building-the-back-end-with-fastapi).

**See it:**

In [ ]:
from web_lab.app import RideCreate
try:
    RideCreate(passenger=' ', pickup='UCC', destination='ucc', distance_km=-1, request_key='x')
except Exception as e:
    print(e)

**Measure it.** Predict first, then run. *What to look for:* Every invalid payload is rejected and the table stays empty.

In [ ]:
res7 = lab(7)

## W08 — State machine

**Concept.** Only the edges of the ride life-cycle are allowed; everything else is 409. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-13--apis-and-rest-design).

**See it:**

In [ ]:
from web_lab.app import TRANSITIONS
for state, nxt in TRANSITIONS.items():
    print(f'{state:<10} -> {sorted(nxt) or "(final)"}')

**Measure it.** Predict first, then run. *What to look for:* The heatmap's 200 cells must match the printed edges exactly.

In [ ]:
res8 = lab(8)

## W09 — Concurrency and idempotency

**Concept.** Simultaneous retries with one idempotency key create one row, thanks to a UNIQUE constraint. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-17--transactions-concurrency-and-python-with-sqlite).

**See it:**

In [ ]:
print([l.strip() for l in Path('web_lab/app.py').read_text().splitlines() if 'request_key TEXT' in l])

**Measure it.** Predict first, then run. *What to look for:* The same-key line stays at 1 for any number of simultaneous requests.

In [ ]:
res9 = lab(9)

## W10 — SQL injection

**Concept.** Concatenated SQL lets input change the query; `?` parameters never do. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-16--sql-queries-joins-aggregation-indexes-and-injection).

**See it:**

In [ ]:
status = "x' OR '1'='1"
print(f"SELECT * FROM rides WHERE status = '{status}'   <- what concatenation builds")

**Measure it.** Predict first, then run. *What to look for:* Concatenation returns every row (or rows of another table); parameters return the honest answer.

In [ ]:
res10 = lab(10)

## W11 — Indexes

**Concept.** An index turns a full scan into a search. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-16--sql-queries-joins-aggregation-indexes-and-injection).

**See it:**

In [ ]:
import sqlite3
from web_lab.app import SCHEMA
db = sqlite3.connect(':memory:'); db.executescript(SCHEMA)
print(db.execute("EXPLAIN QUERY PLAN SELECT * FROM rides WHERE status='requested'").fetchall())
print(db.execute("EXPLAIN QUERY PLAN SELECT * FROM rides WHERE pickup='UCC'").fetchall())

**Measure it.** Predict first, then run. *What to look for:* Log scale: the scan line climbs, the index line stays nearly flat.

In [ ]:
res11 = lab(11)

## W12 — Transactions

**Concept.** BEGIN … COMMIT makes a multi-step change all-or-nothing. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-17--transactions-concurrency-and-python-with-sqlite).

**See it:**

In [ ]:
print('Half of the 100 trials crash between the debit and the credit.')

**Measure it.** Predict first, then run. *What to look for:* Without a transaction, every crash loses money; with one, none do.

In [ ]:
res12 = lab(12)

## W13 — JOIN, GROUP BY, foreign key

**Concept.** Rides refer to drivers by id; queries JOIN to get names and GROUP BY to count. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-15--relational-databases-tables-keys-constraints-and-normalisation).

**See it:**

In [ ]:
from web_lab.app import SCHEMA
print(SCHEMA)

**Measure it.** Predict first, then run. *What to look for:* The note shows the FOREIGN KEY violation message from SQLite.

In [ ]:
res13 = lab(13)

## W14 — Pagination

**Concept.** `limit`/`offset` bound each response; `X-Total-Count` tells the client how many pages exist. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-21--performance-caching-pagination-and-load).

**See it:**

In [ ]:
print('GET /api/rides?limit=20&offset=40  ->  rows 41-60, newest first')

**Measure it.** Predict first, then run. *What to look for:* Bytes grow linearly with the page size.

In [ ]:
res14 = lab(14)

## W15 — ETag caching

**Concept.** `If-None-Match` + an unchanged ETag → 304 with an empty body. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-21--performance-caching-pagination-and-load).

**See it:**

In [ ]:
print('200 (body) -> 304 -> 304 -> data changes -> 200 (new ETag) -> 304')

**Measure it.** Predict first, then run. *What to look for:* Bars at zero are the bytes HTTP caching saved.

In [ ]:
res15 = lab(15)

## W16 — CORS and security headers

**Concept.** Only the allowed development origin passes the preflight; every response carries security headers. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-18--putting-it-together-one-request-through-every-layer).

**See it:**

In [ ]:
from web_lab.app import SECURITY_HEADERS, DEV_ORIGINS
print(DEV_ORIGINS); print(json.dumps(SECURITY_HEADERS, indent=1))

**Measure it.** Predict first, then run. *What to look for:* Row 4 (evil.example) must lack Access-Control-Allow-Origin.

In [ ]:
res16 = lab(16)

## W17 — Latency as data grows

**Concept.** Median and 95th-percentile time of a paginated GET as the table grows. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-21--performance-caching-pagination-and-load).

**See it:**

In [ ]:
print('Percentiles describe the slow tail that averages hide.')

**Measure it.** Predict first, then run. *What to look for:* With pagination and an index, latency grows only slowly.

In [ ]:
res17 = lab(17)

## W18 — Throughput vs concurrency

**Concept.** One Uvicorn worker: more clients raise latency once it is saturated. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-21--performance-caching-pagination-and-load).

**See it:**

In [ ]:
print('Little\'s law: requests in the system = throughput x latency')

**Measure it.** Predict first, then run. *What to look for:* Throughput levels off; latency rises roughly in proportion to the number of clients.

In [ ]:
res18 = lab(18)

## W19 — Browser end-to-end

**Concept.** Playwright clicks through the real page: book, assign, complete, book, delete. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-22--testing-web-applications).

**See it:**

In [ ]:
print('Each step: click -> fetch -> FastAPI -> SQLite -> JSON -> DOM update')

**Measure it.** Predict first, then run. *What to look for:* After the run: `display(Image(OUT/'W19_4_complete_ride.png', width=600))`.

In [ ]:
res19 = lab(19)

## W20 — Async waits

**Concept.** `asyncio.gather` (Python) and `Promise.all` (JavaScript) overlap waiting. [Read the lesson](https://github.com/FUllSCIPTS/ATLAS/blob/main/docs/WEB_TECHNOLOGY.md#lesson-12--asynchronous-javascript-the-event-loop-promises-asyncawait-and-fetch).

**See it:**

In [ ]:
import asyncio, time
async def demo():
    t = time.perf_counter(); await asyncio.gather(*(asyncio.sleep(0.05) for _ in range(5)))
    return (time.perf_counter() - t) * 1000
print(f"five 50 ms waits together took {wl.in_thread(lambda: asyncio.run(demo())):.0f} ms")

**Measure it.** Predict first, then run. *What to look for:* Sequential grows linearly; gather stays near one wait.

In [ ]:
res20 = lab(20)

In [ ]:
if res1: print(res1['extra']['raw_http'])
for name in ('W05_viewport_480.png', 'W05_viewport_1280.png', 'W06_xss_safe_card.png', 'W19_4_complete_ride.png'):
    if (OUT / name).exists():
        print(name); display(Image(filename=str(OUT / name), width=520))

In [ ]:
import shutil
shutil.make_archive('web_technology_results', 'zip', OUT)
print('web_technology_results.zip: CSV, JSON and PNG for every experiment, plus browser screenshots')

## Next steps
Run the site yourself (`python -m uvicorn web_lab.app:app --reload` on your own computer), keep DevTools → Network open, and work through the capstone in the course guide: passenger accounts, login, `GET /api/my-rides` with authorisation checks, keyset pagination and new tests.